# KenLM grid (Google Colab)

kenlm would not build on molab's 3.13, while Colab's 3.11 builds it cleanly. This notebook mounts Drive, installs kenlm, and sweeps alpha/beta over `dev_logits.npz`.

**First, in the molab overnight notebook:** run `push_gdrive(only={"FINAL"})` so that `dev_logits.npz` lands in `CLEAR/Phase 1/runs/FINAL/` on Drive.

Then run the cells in order. For `FINAL_hdo`, change `RUN` in cell 3.

In [ ]:
!pip install -q pyctcdecode pypi-kenlm jiwer "numpy<2"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
RUN = "FINAL"          # change this for FINAL_hdo
NPZ = f"/content/drive/MyDrive/CLEAR/Phase 1/runs/{RUN}/dev_logits.npz"
print("npz present:", os.path.exists(NPZ))
print(NPZ)
if not os.path.exists(NPZ):
    d = os.path.dirname(NPZ)
    print("WARNING: not found. Folder contents:",
          os.listdir(d) if os.path.isdir(d) else '(the folder does not exist either - check the Drive path)')

In [ ]:

# --- KenLM grid engine (identical to kenlm_grid.py, self-contained) ---
import gzip, re, time, urllib.request
from itertools import groupby
from pathlib import Path
import numpy as np, jiwer
from pyctcdecode import build_ctcdecoder

LM_URL = "https://www.openslr.org/resources/11/3-gram.pruned.1e-7.arpa.gz"
CHARS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ'")

def build_vocab():
    v = {c: i for i, c in enumerate(CHARS)}
    v["|"], v["[UNK]"], v["[PAD]"] = len(v), len(v)+1, len(v)+2
    return v

def labels_of(vocab):
    L = [""]*len(vocab)
    for t,i in vocab.items():
        if t=="|":        L[i]=" "
        elif t=="[PAD]":  L[i]=""      # the single CTC blank
        elif t=="[UNK]":  L[i]="\u2047"  # unique; norm() already strips it
        else:             L[i]=t
    return L

_NORM = re.compile(r"[^A-Z' ]+")
def norm(s):
    s = s.upper().replace("|"," ")
    return " ".join(_NORM.sub(" ", s).split())

def greedy(logits, vocab):
    blank, unk = vocab["[PAD]"], vocab["[UNK]"]
    i2c = {i:c for c,i in vocab.items()}
    ids = logits.argmax(-1)
    return "".join(i2c[k] for k,_ in groupby(ids.tolist())
                   if k not in (blank,unk)).replace("|"," ").strip()

def ensure_lm(path="/content/3-gram.pruned.1e-7.arpa"):
    p = Path(path)
    if p.exists(): return str(p)
    gz = p.with_suffix(p.suffix+".gz")
    print("[LM] downloading...")
    urllib.request.urlretrieve(LM_URL, gz)
    with gzip.open(gz,"rb") as f, open(p,"wb") as o: o.write(f.read())
    print("[LM] ready")
    return str(p)

def run_grid(npz_path, alphas=(0.3,0.5,0.7,0.9), betas=(0.5,1.0,1.5,2.0), beam=100):
    z = np.load(npz_path, allow_pickle=True)
    logits = [np.asarray(l, np.float32) for l in (z["logits"] if "logits" in z else z["arr_0"])]
    refs = [norm(str(r)) for r in (z["refs"] if "refs" in z else z["texts"])]
    vocab = build_vocab(); labels = labels_of(vocab)
    g_hyp = [norm(greedy(l, vocab)) for l in logits]
    gw, gc = jiwer.wer(refs,g_hyp), jiwer.cer(refs,g_hyp)
    print(f"[GREEDY] WER {gw*100:.2f}  CER {gc*100:.2f}  W/C {gw/gc:.2f}")
    lm = ensure_lm()
    best = (1e9,None,None,None,None)   # wer, a, b, cer, ratio
    for a in alphas:
        for b in betas:
            dec = build_ctcdecoder(labels, kenlm_model_path=lm, alpha=a, beta=b)
            t0 = time.perf_counter()
            hyp = [norm(dec.decode(l, beam_width=beam)) for l in logits]
            w, c = jiwer.wer(refs, hyp), jiwer.cer(refs, hyp)
            r = w/c if c else 0.0
            if w < best[0]: best = (w,a,b,c,r)
            print(f"  a={a:.2f} b={b:.2f} | WER {w*100:.2f}  CER {c*100:.2f}  W/C {r:.2f}"
                  f" | {time.perf_counter()-t0:.0f}s" + (" *" if w==best[0] else ""))
    print(f"\n[BEST] a={best[1]} b={best[2]} -> WER {best[0]*100:.2f}  "
          f"CER {best[3]*100:.2f}  W/C {best[4]:.2f}")
    return best

run_grid(NPZ)
